# DataQualityAgent — EDA & Cleaning Report

Этот ноутбук закрывает требования задания 2:
- Детект проблем качества: пропуски, дубли, выбросы (IQR), дисбаланс классов
- Визуализация каждой проблемы
- Минимум 2 стратегии чистки и сравнение до/после
- Обоснование выбранной стратегии (Markdown-ячейка)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from data_quality_agent import DataQualityAgent

data_dir = Path('data/raw')
candidates = [
    data_dir / 'merged_with_appendix.parquet',
    data_dir / 'merged_raw.parquet',
    data_dir / 'merged_raw.csv',
]

path = next((p for p in candidates if p.exists()), None)
assert path is not None, f'No dataset found. Looked for: {candidates}'
print('Loading:', path)

if path.suffix == '.parquet':
    df = pd.read_parquet(path)
else:
    df = pd.read_csv(path)

df.head()

In [ ]:
agent = DataQualityAgent()
report = agent.detect_issues(df)
report

## Визуализация проблем

Ниже — минимальные визуализации под требования задания.

In [ ]:
# 1) Missing values
missing = report['missing']
missing_cols = [(k, v['count']) for k, v in missing.items() if k != '_required_rows']
missing_cols = sorted(missing_cols, key=lambda x: x[1], reverse=True)

if missing_cols:
    cols, counts = zip(*missing_cols)
    plt.figure(figsize=(10, 4))
    plt.bar(cols, counts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Missing values by column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values detected (besides optional columns).')


In [ ]:
# 2) Duplicates
dups = report['duplicates']
print('Duplicates count:', dups['count'])
print('Duplicate rate:', dups['rate'])
print('Examples:', dups.get('examples', [])[:3])


In [ ]:
# 3) Outliers (IQR) — visualized on derived text length features
import numpy as np

text = df['text'].astype('string').fillna('')
lens = text.str.len().to_numpy()
q1, q3 = np.quantile(lens, [0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

plt.figure(figsize=(10, 4))
plt.hist(lens, bins=50)
plt.axvline(lo)
plt.axvline(hi)
plt.title('Text length (chars) with IQR bounds')
plt.tight_layout()
plt.show()

print('IQR bounds:', lo, hi)
print('Outliers count:', int(((lens < lo) | (lens > hi)).sum()))


In [ ]:
# 4) Class imbalance
imb = report['imbalance']
print('Label column:', imb.get('label_col'))
counts = imb.get('counts', {})
if counts:
    items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    labels, cts = zip(*items)
    plt.figure(figsize=(10, 4))
    plt.bar(labels, cts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Class distribution (label)')
    plt.tight_layout()
    plt.show()
else:
    print('No label distribution available.')


## Часть 2: Хирург — стратегии чистки

Ниже две стратегии чистки.

**Strategy A (консервативная):**
- missing: fill
- duplicates: drop
- outliers: drop_iqr

**Strategy B (агрессивная по пропускам + иначе по дублям):**
- missing: drop
- duplicates: keep_longest
- outliers: drop_iqr


In [ ]:
strategy_a = {'missing': 'fill', 'duplicates': 'drop', 'outliers': 'drop_iqr'}
strategy_b = {'missing': 'drop', 'duplicates': 'keep_longest', 'outliers': 'drop_iqr'}

df_a = agent.fix(df, strategy=strategy_a)
df_b = agent.fix(df, strategy=strategy_b)

comp_a = agent.compare(df, df_a)
comp_b = agent.compare(df, df_b)

import pandas as pd
print('--- Strategy A comparison ---')
pd.DataFrame(comp_a['table'])


In [ ]:
print('--- Strategy B comparison ---')
pd.DataFrame(comp_b['table'])


## Часть 3: Аргумент — почему выбранная стратегия лучше

*(Заполни этот блок под свою ML-задачу.)*

Пример аргументации для RuFPBench-MVP:
- Мы предпочитаем **Strategy A**, потому что она сохраняет больше данных и не выбрасывает строки из-за пропусков в необязательных полях.
- Дубли удаляются, чтобы не раздувать частоты отдельных формулировок.
- Выбросы по длине текста удаляются (drop_iqr), потому что очень длинные записи часто являются артефактами (склейки/HTML/лог-дампы) и мешают дальнейшей авторазметке.
